In [0]:
%sql
-- Gold Layer: Patient Encounter Summary
-- Aggregates each patient's complete encounter history into a single row

CREATE OR REPLACE TABLE gold.patient_encounter_summary AS
SELECT
    -- Patient identifiers and demographics
    p.patient_id,
    p.mrn,
    p.first_name,
    p.last_name,
    p.age_years,
    p.age_group,
    p.gender,
    p.zip_code,
    
    -- Encounter volume metrics
    COUNT(DISTINCT e.encounter_id) as total_encounters,
    SUM(e.is_inpatient) as inpatient_encounters,
    SUM(e.is_emergency) as emergency_encounters,
    SUM(e.is_outpatient) as outpatient_encounters,
    SUM(e.is_observation) as observation_encounters,
    
    -- Inpatient utilization
    SUM(CASE WHEN e.is_inpatient = 1 THEN e.length_of_stay_days ELSE 0 END) as total_inpatient_days,
    ROUND(
        AVG(CASE WHEN e.is_inpatient = 1 THEN e.length_of_stay_days END),
        2
    ) as avg_inpatient_los,
    
    -- Most recent activity
    MAX(e.discharge_datetime) as last_encounter_date,
    DATEDIFF(day, MAX(e.discharge_datetime), CURRENT_DATE()) as days_since_last_encounter,
    
    -- Clinical complexity
    COUNT(DISTINCT d.icd10_code) as unique_diagnoses,
    COUNT(DISTINCT d.diagnosis_category) as unique_diagnosis_categories,
    COUNT(DISTINCT pr.procedure_code) as total_procedures,
    
    -- Most common department
    MODE() WITHIN GROUP (ORDER BY e.department) as most_frequent_department,
    
    -- Weekend utilization
    SUM(e.is_weekend_admission) as weekend_admissions,
    ROUND(
        SUM(e.is_weekend_admission) * 100.0 / NULLIF(COUNT(e.encounter_id), 0),
        1
    ) as weekend_admission_rate,
    
    -- Utilization category
    CASE 
        WHEN COUNT(DISTINCT e.encounter_id) >= 10 THEN 'High Utilizer'
        WHEN COUNT(DISTINCT e.encounter_id) >= 5 THEN 'Moderate Utilizer'
        WHEN COUNT(DISTINCT e.encounter_id) >= 3 THEN 'Average Utilizer'
        ELSE 'Low Utilizer'
    END as utilization_category,
    
    -- First and last encounter dates
    MIN(e.admit_datetime) as first_encounter_date,
    DATEDIFF(day, MIN(e.admit_datetime), MAX(e.discharge_datetime)) as days_as_patient,
    
    -- Metadata
    CURRENT_TIMESTAMP() as _created_at

FROM silver.patients p
LEFT JOIN silver.encounters e ON p.patient_id = e.patient_id
LEFT JOIN silver.diagnoses d ON e.encounter_id = d.encounter_id AND d.is_primary_diagnosis = 1
LEFT JOIN silver.procedures pr ON e.encounter_id = pr.encounter_id
GROUP BY 
    p.patient_id, p.mrn, p.first_name, p.last_name, 
    p.age_years, p.age_group, p.gender, p.zip_code;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify Gold table was created successfully
SELECT 
    '🎯 GOLD LAYER - PATIENT SUMMARY' as section,
    '=' as separator,
    '=' as value
UNION ALL
SELECT
    '',
    'Total Patients',
    CAST(COUNT(*) AS STRING)
FROM gold.patient_encounter_summary
UNION ALL
SELECT
    '',
    'Avg Encounters per Patient',
    CAST(ROUND(AVG(total_encounters), 1) AS STRING)
FROM gold.patient_encounter_summary
UNION ALL
SELECT
    '',
    'Patients with Inpatient Stays',
    CAST(COUNT(*) AS STRING)
FROM gold.patient_encounter_summary
WHERE inpatient_encounters > 0
UNION ALL
SELECT
    '',
    '',
    ''
UNION ALL
SELECT
    '📊 UTILIZATION CATEGORIES',
    '',
    ''
UNION ALL
SELECT
    '',
    utilization_category,
    CONCAT(
        CAST(COUNT(*) AS STRING),
        ' (',
        CAST(ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS STRING),
        '%)'
    )
FROM gold.patient_encounter_summary
GROUP BY utilization_category;

section,separator,value
🎯 GOLD LAYER - PATIENT SUMMARY,=,=
,Avg Encounters per Patient,4.1
,Patients with Inpatient Stays,700
,,
📊 UTILIZATION CATEGORIES,,
,Low Utilizer,204 (20.4%)
,High Utilizer,3 (0.3%)
,Average Utilizer,431 (43.1%)
,Moderate Utilizer,362 (36.2%)
,Total Patients,1000


In [0]:
%sql
-- Identify high-utilizer patients (care management candidates)
SELECT 
    mrn,
    CONCAT(first_name, ' ', last_name) as patient_name,
    age_years,
    age_group,
    total_encounters,
    inpatient_encounters,
    emergency_encounters,
    ROUND(total_inpatient_days, 1) as total_inpatient_days,
    ROUND(avg_inpatient_los, 1) as avg_los,
    most_frequent_department,
    unique_diagnoses,
    days_since_last_encounter
FROM gold.patient_encounter_summary
WHERE utilization_category = 'High Utilizer'
ORDER BY total_encounters DESC, total_inpatient_days DESC
LIMIT 10;

mrn,patient_name,age_years,age_group,total_encounters,inpatient_encounters,emergency_encounters,total_inpatient_days,avg_los,most_frequent_department,unique_diagnoses,days_since_last_encounter
MRN000070,SUSAN JACKSON,28,Adult,10,16,5,98.6,6.2,Obstetrics,6,24
MRN000847,KAREN RODRIGUEZ,47,Adult,10,5,2,23.0,4.6,Surgery,9,25
MRN000136,JESSICA GONZALEZ,24,Adult,10,8,4,18.3,2.3,Emergency,5,11


In [0]:
%sql
-- Patient utilization patterns by age group
SELECT 
    age_group,
    COUNT(*) as patient_count,
    ROUND(AVG(total_encounters), 1) as avg_encounters,
    ROUND(AVG(inpatient_encounters), 1) as avg_inpatient_visits,
    ROUND(AVG(emergency_encounters), 1) as avg_ed_visits,
    ROUND(AVG(total_inpatient_days), 1) as avg_total_inpatient_days,
    ROUND(AVG(unique_diagnoses), 1) as avg_diagnoses,
    SUM(CASE WHEN utilization_category = 'High Utilizer' THEN 1 ELSE 0 END) as high_utilizers,
    ROUND(
        SUM(CASE WHEN utilization_category = 'High Utilizer' THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) as high_utilizer_rate
FROM gold.patient_encounter_summary
GROUP BY age_group
ORDER BY 
    CASE age_group
        WHEN 'Pediatric' THEN 1
        WHEN 'Adult' THEN 2
        WHEN 'Senior' THEN 3
    END;

age_group,patient_count,avg_encounters,avg_inpatient_visits,avg_ed_visits,avg_total_inpatient_days,avg_diagnoses,high_utilizers,high_utilizer_rate
Adult,593,4.0,4.5,2.0,24.1,3.4,3,0.5
Senior,407,4.1,4.7,2.0,24.9,3.6,0,0.0


In [0]:
%sql
-- Which departments serve which patient populations?
SELECT 
    most_frequent_department as department,
    COUNT(*) as patients_primarily_served,
    ROUND(AVG(total_encounters), 1) as avg_encounters_per_patient,
    ROUND(AVG(total_inpatient_days), 1) as avg_total_inpatient_days,
    ROUND(AVG(age_years), 1) as avg_patient_age,
    COUNT(CASE WHEN utilization_category IN ('High Utilizer', 'Moderate Utilizer') THEN 1 END) as high_moderate_utilizers
FROM gold.patient_encounter_summary
WHERE most_frequent_department IS NOT NULL
GROUP BY most_frequent_department
ORDER BY patients_primarily_served DESC;

department,patients_primarily_served,avg_encounters_per_patient,avg_total_inpatient_days,avg_patient_age,high_moderate_utilizers
Cardiology,145,4.2,24.4,55.3,55
ICU,125,4.2,26.5,57.0,52
Emergency,123,4.0,23.9,59.9,44
Neurology,102,3.9,28.5,58.9,40
Medicine,102,3.8,21.9,58.7,37
Oncology,96,4.1,24.7,58.4,35
Obstetrics,94,4.1,23.6,60.0,30
Orthopedics,82,3.9,26.2,54.3,23
Surgery,67,4.4,21.8,55.5,33
Pediatrics,64,3.7,19.9,54.4,16


In [0]:
%sql
-- Quality checks for Gold table
SELECT 
    'GOLD LAYER QUALITY CHECKS' as check_name,
    '' as status
UNION ALL
SELECT
    'All Silver patients in Gold',
    CASE 
        WHEN (SELECT COUNT(*) FROM silver.patients) = 
             (SELECT COUNT(*) FROM gold.patient_encounter_summary)
        THEN '✓ PASS (1,000 = 1,000)'
        ELSE CONCAT('✗ FAIL (', 
                    CAST((SELECT COUNT(*) FROM silver.patients) AS STRING),
                    ' ≠ ',
                    CAST((SELECT COUNT(*) FROM gold.patient_encounter_summary) AS STRING),
                    ')')
    END
UNION ALL
SELECT
    'No negative encounter counts',
    CASE 
        WHEN (SELECT MIN(total_encounters) FROM gold.patient_encounter_summary) >= 0
        THEN '✓ PASS'
        ELSE '✗ FAIL'
    END
UNION ALL
SELECT
    'All utilization categories assigned',
    CASE
        WHEN (SELECT COUNT(*) FROM gold.patient_encounter_summary WHERE utilization_category IS NULL) = 0
        THEN '✓ PASS'
        ELSE '✗ FAIL'
    END
UNION ALL
SELECT
    'High utilizers exist',
    CASE
        WHEN (SELECT COUNT(*) FROM gold.patient_encounter_summary WHERE utilization_category = 'High Utilizer') > 0
        THEN CONCAT('✓ PASS (', 
                    CAST((SELECT COUNT(*) FROM gold.patient_encounter_summary WHERE utilization_category = 'High Utilizer') AS STRING),
                    ' patients)')
        ELSE '⚠ WARNING'
    END
UNION ALL
SELECT
    'Average encounters reasonable',
    CASE
        WHEN (SELECT AVG(total_encounters) FROM gold.patient_encounter_summary) BETWEEN 1 AND 20
        THEN CONCAT('✓ PASS (Avg: ', 
                    CAST(ROUND((SELECT AVG(total_encounters) FROM gold.patient_encounter_summary), 1) AS STRING),
                    ')')
        ELSE '⚠ CHECK'
    END;

check_name,status
GOLD LAYER QUALITY CHECKS,
All Silver patients in Gold,"✓ PASS (1,000 = 1,000)"
No negative encounter counts,✓ PASS
All utilization categories assigned,✓ PASS
High utilizers exist,✓ PASS (3 patients)
Average encounters reasonable,✓ PASS (Avg: 4.1)


In [0]:
%sql
-- Complete Pipeline Summary
SELECT 
    '🏗️  COMPLETE DATA PIPELINE STATUS' as layer,
    '=' as table_name,
    '=' as records
UNION ALL
SELECT
    '',
    'Layer',
    'Status'
UNION ALL
SELECT
    '📁 BRONZE (Raw)',
    'patients',
    CAST((SELECT COUNT(*) FROM bronze.patients) AS STRING)
UNION ALL
SELECT
    '',
    'encounters',
    CAST((SELECT COUNT(*) FROM bronze.encounters) AS STRING)
UNION ALL
SELECT
    '',
    'diagnoses',
    CAST((SELECT COUNT(*) FROM bronze.diagnoses) AS STRING)
UNION ALL
SELECT
    '',
    'procedures',
    CAST((SELECT COUNT(*) FROM bronze.procedures) AS STRING)
UNION ALL
SELECT
    '',
    'Bronze Total',
    CAST(
        (SELECT COUNT(*) FROM bronze.patients) +
        (SELECT COUNT(*) FROM bronze.encounters) +
        (SELECT COUNT(*) FROM bronze.diagnoses) +
        (SELECT COUNT(*) FROM bronze.procedures)
    AS STRING)
UNION ALL
SELECT '', '', ''
UNION ALL
SELECT
    '🔧 SILVER (Cleaned)',
    'patients',
    CAST((SELECT COUNT(*) FROM silver.patients) AS STRING)
UNION ALL
SELECT
    '',
    'encounters',
    CAST((SELECT COUNT(*) FROM silver.encounters) AS STRING)
UNION ALL
SELECT
    '',
    'diagnoses',
    CAST((SELECT COUNT(*) FROM silver.diagnoses) AS STRING)
UNION ALL
SELECT
    '',
    'procedures',
    CAST((SELECT COUNT(*) FROM silver.procedures) AS STRING)
UNION ALL
SELECT
    '',
    'Silver Total',
    CAST(
        (SELECT COUNT(*) FROM silver.patients) +
        (SELECT COUNT(*) FROM silver.encounters) +
        (SELECT COUNT(*) FROM silver.diagnoses) +
        (SELECT COUNT(*) FROM silver.procedures)
    AS STRING)
UNION ALL
SELECT '', '', ''
UNION ALL
SELECT
    '📊 GOLD (Analytics)',
    'patient_encounter_summary',
    CAST((SELECT COUNT(*) FROM gold.patient_encounter_summary) AS STRING)
UNION ALL
SELECT '', '', ''
UNION ALL
SELECT
    '✅ PIPELINE STATUS',
    'Bronze → Silver → Gold',
    'COMPLETE';

layer,table_name,records
🏗️ COMPLETE DATA PIPELINE STATUS,=,=
,Layer,Status
📁 BRONZE (Raw),patients,1000
,encounters,4053
,diagnoses,12156
,procedures,8916
,Bronze Total,26125
,,
🔧 SILVER (Cleaned),patients,1000
,encounters,4053
